#### Environment Check

In [1]:
import sys
print(sys.executable)

/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/.venv/bin/python


#### Setup

In [2]:
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

#### Project Paths

In [3]:
LAB_DIR = Path("..").resolve()
CODE_DIR = LAB_DIR / "code"
DATA_DIR = LAB_DIR / "data"

str(LAB_DIR), str(DATA_DIR)

('/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab',
 '/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab/data')

#### Import Course Helpers

In [4]:
import sys

sys.path.append(str(CODE_DIR))

from ingest import load_faq_data, build_index
from evaluation_utils import (
    compute_relevance,
    compute_relevance_total,
    hit_rate,
    mrr,
    evaluate,
)

#### Load Ground Truth

In [5]:
ground_truth_path = DATA_DIR / "ground_truth-new.csv"

df_ground_truth = pd.read_csv(ground_truth_path)

df_ground_truth.head()

,question,document
0,What is the recommended way to begin the cours...,04919992b3
1,Which course resources should I open first if ...,04919992b3
2,"How do the lectures, notebooks, and homework u...",04919992b3
3,Where can I find the deadlines and submit my h...,04919992b3
4,"Can I start the course at any time, and does t...",04919992b3


#### Convert Ground Truth To Records

In [6]:
ground_truth = df_ground_truth.to_dict(orient="records")

len(ground_truth)

565

#### Load FAQ Documents

In [7]:
documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm

len(documents)

113

#### Build Search Index

In [8]:
index = build_index(documents)

#### Define Text Search

In [9]:
def text_search(query):
    boost_dict = {
        "question": 3.0,
        "section": 0.5,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

#### Inspect One Ground Truth Record

In [10]:
q = ground_truth[0]

q

{'question': 'What is the recommended way to begin the course and organize my study each week?',
 'document': '04919992b3'}

#### Search One Question

In [11]:
doc_id = q["document"]

results = text_search(query=q["question"])

for doc in results:
    print(f'{doc["id"]} == {doc_id}: {doc["id"] == doc_id}')

04919992b3 == 04919992b3: True
489dd1c9d9 == 04919992b3: False
ee43413718 == 04919992b3: False
db78580409 == 04919992b3: False
d65e05bd7a == 04919992b3: False


#### Build One Relevance List

In [12]:
relevance = []

for doc in results:
    relevance.append(int(doc["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

#### Use Helper For One Question

In [13]:
compute_relevance(q, text_search)

[1, 0, 0, 0, 0]

#### Evaluate First 15 Questions

In [14]:
relevance_total = []

for q in tqdm(ground_truth[:15]):
    relevance = compute_relevance(q, text_search)
    relevance_total.append(relevance)

relevance_total


  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

#### Hit Rate On First 15

In [15]:
hit_rate(relevance_total)

0.8

#### MRR On First 15

In [16]:
mrr(relevance_total)

0.5466666666666666

#### Evaluate All Questions Manually

In [17]:
relevance_total = compute_relevance_total(
    ground_truth,
    text_search,
)

len(relevance_total)

  0%|          | 0/565 [00:00<?, ?it/s]

565

#### Search Metrics

In [18]:
search_metrics = {
    "hit_rate": hit_rate(relevance_total),
    "mrr": mrr(relevance_total),
}

search_metrics

{'hit_rate': 0.7734513274336283, 'mrr': 0.6204424778761058}

#### Use Generic Evaluate Function

In [19]:
evaluate(
    ground_truth,
    text_search,
)

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.7734513274336283, 'mrr': 0.6204424778761058}

#### Save Search Metrics

In [20]:
REPORTS_DIR = LAB_DIR / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

report_path = REPORTS_DIR / "search_metrics.md"

with open(report_path, "w") as f:
    f.write("# Search Evaluation Metrics\n\n")
    f.write("## Baseline Text Search\n\n")
    f.write(f"- Hit Rate: {search_metrics['hit_rate']}\n")
    f.write(f"- MRR: {search_metrics['mrr']}\n")

report_path

PosixPath('/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab/reports/search_metrics.md')